# Triaxial compression, TWO-PISTON — strain sweep

Analysis of **every** applied-strain level of a `triaxial_compression_two_pist` sweep (drained consolidation at
constant bath pressure: NPT-pistons on both reservoirs, load piston loading the network), overlaid so the whole
stress–strain response is visible (M, G, D_c, κ vs strain) — the twelve figures of `triaxial_compression_sweep.ipynb`
plus the per-level **wet-piston bath check** and **solvent expelled**.  `COMP_LEVELS` must match
`STRAIN_TARGETS=(...)` in `triaxial_compression_two_pist.batch`.  To look closely at one level use
`triaxial_compression_single_two_pist.ipynb`.  Notes at the end (2026-09-16).

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: triaxial.py (all analysis code) + volfrac.py
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
tri = importlib.reload(tri)            # pick up edits to lib/triaxial.py without a kernel restart
tri.setup_style()
print('analysis code: ', LIB / 'triaxial.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
cfg = tri.Config(
    DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,           # <steps> tag = each level's auto-sized HOLD LENGTH; None resolves it from the files
    RUN_ID      = "periodic_rho04_14M_9M_twoPlatesMove_7_sweep",  # local folder under flow_data_local/{compression,plots}
    COMP_LEVELS = ["0.10", "0.15", "0.20", "0.3", "0.4", "0.5", "0.7"],   # = STRAIN_TARGETS=(...) in triaxial_compression_two_pist.batch
    mode        = "compression",
    two_pist    = True,           # Expanse folder triaxial_compression_two_pist; syncs the wet-piston files
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, P_CAL = 1.5,
    M_SUBTRACT_REF = True, G_SUBTRACT_REF = True,
    DC_FREE_AMPS = True, DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_SLOW_REF = 0.17, DC_TARGET_RESID = 0.01,
)

In [ ]:
SYNC, FORCE_SYNC = True, False
if SYNC:
    tri.sync_from_expanse(cfg, force=FORCE_SYNC)

In [ ]:
R = tri.load_reference(cfg)
LEVELS = [L for L in (tri.load_level(cfg, R, lvl) for lvl in cfg.COMP_LEVELS) if L is not None]
assert LEVELS, 'no level loaded -- run the sync cell'
LEVELS.sort(key=lambda L: L['eps'])
tri.add_volume_fractions(cfg, R, LEVELS)
tri.print_summary(cfg, LEVELS)

## 2 · Figures

In [ ]:
# 1 · Strain diagnostic, all levels
tri.fig_strain(cfg, R, LEVELS, stem='sweep_strain_diagnostic');

In [ ]:
# 2 · Solvent volume fraction φ_s: reference dashed + one plateau curve per level
tri.fig_volfrac_sweep(cfg, R, LEVELS);

In [ ]:
# 3 · Total stress / P_bath evolution, all levels (feed-reservoir pore baseline)
tri.fig_total_stress_sweep(cfg, R, LEVELS);

In [ ]:
# 4 · Solvent partial | polymer partial | total σ_zz evolutions, all levels
tri.fig_partial_stress_sweep(cfg, R, LEVELS);

In [ ]:
# 5 · Network stress σ'_zz, σ'_xx, σ'_yy: FINAL equilibrated state of every level + reference (no evolution curves:
#     p_pore is not uniform while the gel consolidates)
tri.fig_network_stress_sweep(cfg, R, LEVELS);

In [ ]:
# 6 · Load-piston pressure histories, linear + log, all levels
tri.fig_piston_sweep(cfg, R, LEVELS);

In [ ]:
# 7 · Longitudinal modulus M vs applied strain (network vs load piston)
tri.fig_M_sweep(cfg, R, LEVELS);

In [ ]:
# 7b · Stress vs strain with the ε = 0 readings; least-squares slopes = offset-free M
tri.fig_stress_strain_sweep(cfg, R, LEVELS);

In [ ]:
# 8 · Network-stress anisotropy vs step, all levels
tri.fig_ratio_sweep(cfg, R, LEVELS);

In [ ]:
# 9 · Shear modulus G vs applied strain
tri.fig_G_sweep(cfg, R, LEVELS);

In [ ]:
# 10 · D_c vs applied strain + the consolidation fit of every level
tri.fig_Dc_sweep(cfg, R, LEVELS);

In [ ]:
# 11 · κ = D_c/M vs applied strain
tri.fig_kappa_sweep(cfg, R, LEVELS);

In [ ]:
# 12 · TWO-PISTON: (a) bath check per level (feed solid, permeate dashed) vs P_target; (b) solvent expelled, cumulative
tri.fig_wet_pistons_sweep(cfg, R, LEVELS);

In [ ]:
# 13 · Thermodynamic pressure P_th = −⅓ tr(σ^t) evolution, all levels (dotted = P_bath)
tri.fig_thermo_pressure_sweep(cfg, R, LEVELS);

In [ ]:
# 14 · Osmotic pressure Π = −⅓ tr(σ′): final equilibrated state of every level + reference
tri.fig_osmotic_pressure_sweep(cfg, R, LEVELS);


## Notes

Two-piston specifics are described in `triaxial_compression_single_two_pist.ipynb` (geometry, feed-reservoir
pore baseline, `[load | feed | perm]` piston columns, bath check).  Everything else — the eleven figures, M, G,
D_c, κ and the sweep conventions — is unchanged from `triaxial_compression_sweep.ipynb`.  The sweep is the only
sweep in the two-piston sequence; deeper levels need a data file built with a larger `margin_feed`
(`slab_two_pistons.ipynb` prints the strain the feed-side margin can absorb).